TASK B (Road Accidents)

In this task, the aim is to investigate road accidents as events occurring on a spatial road network rather than as points in unrestricted planar space. This follows the spatial-network perspective taught in the module: roads are represented as edges, intersections as nodes, and events such as accidents can be analysed relative to the network structure itself. In particular, the module materials emphasise that point-pattern analysis on networks should use network distance along edges, and that spatial autocorrelation for line-based attributes can be defined using edge adjacency / contiguity. 

This notebook answers the three Task B questions:

1. Visualise the distribution of multi-year accident locations on the road network.
2. Test whether roads with many accidents tend to connect to other roads with many accidents, using:
   - a network K-function for clustering over network distance,
   - Moran’s I for edge-level spatial autocorrelation.
3. Investigate whether accidents occur closer to intersections or partway along road segments, by measuring the fraction of segment length from the nearest intersection. 

The notebook is designed in the same practical style as the labs: load data, reconstruct the study area, map events to the road network, compute the required statistics, and interpret the outputs clearly.

In [1]:
# Uncomment if needed the first time you run this notebook:
# %pip install osmnx geopandas libpysal esda

import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import osmnx as ox

from shapely.geometry import Point, box
from collections import defaultdict
from itertools import combinations

from libpysal.weights import W
from esda.moran import Moran

This establishes the environment for rebuilding the Task A study area, attaching accident events to road segments, and carrying out the network-based clustering and autocorrelation analyses.

Step 1: Rebuild the study area from Task A

To keep the workflow reproducible, Task B uses exactly the same study area that was selected in Task A. This is important because the road-accident analysis should be based on the same 1 km² road network that was previously justified as both central and accident-dense.

The final Task A study box was:

- Easting: 429934 to 430934
- Northing: 433255 to 434255

This ensures that Task B remains consistent with Task A and can be run independently if needed.

In [2]:
# Fixed study area from Task A
min_easting = 429934
max_easting = 430934
min_northing = 433255
max_northing = 434255

grid_size = 1000
study_box = box(min_easting, min_northing, max_easting, max_northing)

print("Study area rebuilt from Task A.")
print("Easting :", min_easting, "to", max_easting)
print("Northing:", min_northing, "to", max_northing)
print("Area (km²):", (grid_size * grid_size) / 1_000_000)

Study area rebuilt from Task A.
Easting : 429934 to 430934
Northing: 433255 to 434255
Area (km²): 1.0


The same study area has now been reconstructed. This means all later Task B results are directly comparable with the Task A network characteristics.

Step 2: Load and combine the multi-year accident datasets

The coursework specifies that accident analysis should be aggregated across multiple years. Therefore, the 2014, 2015, and 2016 accident datasets are concatenated before filtering to the selected study area. This avoids treating a single year as representative when the question explicitly asks for a multi-year view. 

In [3]:
df14 = pd.read_csv("../../data/raw/accidents_2014.csv", encoding="latin-1")
df15 = pd.read_csv("../../data/raw/accidents_2015.csv", encoding="latin-1")
df16 = pd.read_csv("../../data/raw/accidents_2016.csv", encoding="latin-1")

accidents = pd.concat([df14, df15, df16], ignore_index=True)

print("Total accidents across all years:", len(accidents))
accidents.head()

Total accidents across all years: 7746


,Reference Number,Grid Ref: Easting,Grid Ref: Northing,Number of Vehicles,Accident Date,Time (24hr),1st Road Class,Road Surface,Lighting Conditions,Weather Conditions,Casualty Class,Unnamed: 11,Casualty Severity,Sex of Casualty,Age of Casualty,Type of Vehicle,Expr1
0,1112091,429175,431904,2,01-Jan-14,1840,Unclassified,Wet / Damp,Darkness: street lights present and lit,Raining without high winds,1,Driver,Slight,Male,58,Car,NaN
1,1180869,430429,431025,3,08-Jan-14,1430,Motorway,Wet / Damp,Daylight: street lights present,Fine without high winds,1,Driver,Slight,Male,69,Car,NaN
2,1180869,430429,431025,3,08-Jan-14,1430,Motorway,Wet / Damp,Daylight: street lights present,Fine without high winds,1,Driver,Slight,Female,41,Car,NaN
3,11A0238,424660,427582,3,10-Jan-14,817,Motorway,Dry,Daylight: street lights present,Fine without high winds,1,Driver,Slight,Male,35,Car,NaN
4,11A0238,424660,427582,3,10-Jan-14,817,Motorway,Dry,Daylight: street lights present,Fine without high winds,1,Driver,Slight,Male,25,Car,NaN


In [4]:
# Convert to spatial points in British National Grid
acc_gdf = gpd.GeoDataFrame(
    accidents.copy(),
    geometry=gpd.points_from_xy(
        accidents["Grid Ref: Easting"],
        accidents["Grid Ref: Northing"]
    ),
    crs="EPSG:27700"
)

# Filter to the Task A study area
study_accidents = acc_gdf.cx[min_easting:max_easting, min_northing:max_northing].copy()
study_accidents = study_accidents.reset_index(drop=True)
study_accidents["accident_id"] = np.arange(len(study_accidents))

print("Accidents in study area:", len(study_accidents))
study_accidents.head()

Accidents in study area: 313


,Reference Number,Grid Ref: Easting,Grid Ref: Northing,Number of Vehicles,Accident Date,Time (24hr),1st Road Class,Road Surface,Lighting Conditions,Weather Conditions,Casualty Class,Unnamed: 11,Casualty Severity,Sex of Casualty,Age of Casualty,Type of Vehicle,Expr1,geometry,accident_id
0,12M0822,430373,433481,1,22-Feb-14,1245,Unclassified,Dry,Daylight: street lights present,Fine without high winds,3,Pedestrian,Serious,Female,86,Bus or coach (17 or more passenger seats),NaN,POINT (430373 433481),0
1,12O0439,430066,433771,1,24-Feb-14,1036,Unclassified,Dry,Daylight: street lights present,Fine without high winds,3,Pedestrian,Slight,Male,59,Goods vehicle over 3.5 tonnes and under 7.5 to...,NaN,POINT (430066 433771),1
2,12O1421,430634,433453,2,24-Feb-14,1835,Unclassified,Dry,Darkness: street lights present and lit,Fine without high winds,2,Passenger,Slight,Female,20,Taxi/Private hire car,NaN,POINT (430634 433453),2
3,12R0675,430717,434135,1,27-Feb-14,1300,Unclassified,Dry,Daylight: street lights present,Fine without high winds,1,Driver,Slight,Male,18,Motorcycle over 125cc and up to 500cc,NaN,POINT (430717 434135),3
4,12R0675,430717,434135,1,27-Feb-14,1300,Unclassified,Dry,Daylight: street lights present,Fine without high winds,3,Pedestrian,Slight,Male,19,Motorcycle over 125cc and up to 500cc,NaN,POINT (430717 434135),4


The accident data has now been restricted to the selected 1 km² study area. This gives the exact event set that will be analysed on the corresponding road network in the next steps.

Step 3: Rebuild the drivable road network for the study area

Task B is explicitly “based on the construction of your network,” so the road graph must match the same drivable road network used in Task A. This is reconstructed from OpenStreetMap using the study polygon and network_type='drive', which restricts the network to roads used for driving. This matches the coursework requirement to focus on road traffic accidents on the road network rather than pedestrian infrastructure. 

In [5]:
# Convert study polygon to WGS84 for OSMnx
study_box_gdf = gpd.GeoDataFrame(geometry=[study_box], crs="EPSG:27700").to_crs(epsg=4326)
study_polygon_wgs84 = study_box_gdf.geometry.iloc[0]

# Download drivable road network and project back to metric CRS
G = ox.graph_from_polygon(
    study_polygon_wgs84,
    network_type="drive",
    simplify=True,
    retain_all=False,
    truncate_by_edge=False
)

G_proj = ox.project_graph(G)
Gu = ox.convert.to_undirected(G_proj)

print("Road network rebuilt.")
print("Nodes:", Gu.number_of_nodes())
print("Edges:", Gu.number_of_edges())

Road network rebuilt.
Nodes: 204
Edges: 283


In [6]:
# Convert graph to GeoDataFrames
nodes_gdf, edges_gdf = ox.graph_to_gdfs(Gu, nodes=True, edges=True, fill_edge_geometry=True)

nodes_gdf = nodes_gdf.reset_index()
edges_gdf = edges_gdf.reset_index()

# Unique edge identifier for Task B analysis
edges_gdf["edge_id"] = np.arange(len(edges_gdf))

print("Edges available for analysis:", len(edges_gdf))
edges_gdf.head()

Edges available for analysis: 283


,u,v,key,osmid,highway,lanes,maxspeed,name,oneway,ref,reversed,length,from,to,geometry,junction,access,tunnel,bridge,edge_id
0,611057,6047609602,0,229127796,trunk,3,30 mph,East Street,True,A61,False,32.104149,611057,6047609602,"LINESTRING (596569.662 5961701.725, 596557.934...",NaN,NaN,NaN,NaN,0
1,611057,955003902,0,5946791,trunk_link,2,30 mph,East Street,True,NaN,False,8.103607,955003902,611057,"LINESTRING (596570.849 5961693.703, 596569.662...",NaN,NaN,NaN,NaN,1
2,611057,1606959387,0,229127798,trunk,2,30 mph,East Street,True,A61,False,5.198383,1606959387,611057,"LINESTRING (596573.096 5961697.811, 596569.662...",NaN,NaN,NaN,NaN,2
3,6047609602,26661423,0,778025047,tertiary,1,30 mph,East Street,True,NaN,False,39.978823,6047609602,26661423,"LINESTRING (596557.934 5961731.643, 596553 596...",NaN,NaN,NaN,NaN,3
4,6047609602,682264890,0,"[778025048, 29186597]",trunk,2,30 mph,East Street,True,A61,False,35.009020,6047609602,682264890,"LINESTRING (596557.934 5961731.643, 596558.193...",NaN,NaN,NaN,NaN,4


The Task B road network is now available as both a graph and a set of road-segment geometries. This is important because the later analysis treats accidents as events attached to road segments, not simply as free-standing planar points. The module materials explicitly note that in spatial network analysis, line objects such as road segments can carry attributes such as counts of incidents.